In [20]:
from pathlib import Path
import duckdb

In [21]:
data_dir = Path("/media/datasets/fineweb_10BT")
num_samples = 10

In [22]:
parquet_files = list(data_dir.rglob("*.parquet"))
glob_pattern = str(data_dir / "**" / "*.parquet")
print(glob_pattern)
parquet_files

/media/datasets/fineweb_10BT/**/*.parquet


[PosixPath('/media/datasets/fineweb_10BT/sample/10BT/009_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/014_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/001_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/002_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/008_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/006_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/011_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/004_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/007_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/003_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/013_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/005_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10BT/sample/10BT/000_00000.parquet'),
 PosixPath('/media/datasets/fineweb_10

In [ ]:
con = duckdb.connect()

print("=" * 60)
print("DATASET SCHEMA & TYPES")
print("=" * 60)
schema_df = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{glob_pattern}')"
).pl()
print(schema_df[["column_name", "column_type"]])

1. DATASET SCHEMA & TYPES
shape: (9, 2)
┌────────────────┬─────────────┐
│ column_name    ┆ column_type │
│ ---            ┆ ---         │
│ str            ┆ str         │
╞════════════════╪═════════════╡
│ text           ┆ VARCHAR     │
│ id             ┆ VARCHAR     │
│ dump           ┆ VARCHAR     │
│ url            ┆ VARCHAR     │
│ date           ┆ VARCHAR     │
│ file_path      ┆ VARCHAR     │
│ language       ┆ VARCHAR     │
│ language_score ┆ DOUBLE      │
│ token_count    ┆ BIGINT      │
└────────────────┴─────────────┘


In [ ]:
print("\n" + "=" * 60)
print("GLOBAL DATASET AGGREGATES")
print("=" * 60)
stats_query = f"""
    SELECT 
        COUNT(*) AS total_documents,
        ROUND(AVG(LENGTH(text)), 2) AS avg_char_len,
        MEDIAN(LENGTH(text)) AS median_char_len,
        MIN(LENGTH(text)) AS min_char_len,
        MAX(LENGTH(text)) AS max_char_len,
        -- Rough word estimate (~5 chars per word)
        ROUND(SUM(LENGTH(text)) / 5.0 / 1e9, 3) AS approx_billion_words
    FROM read_parquet('{glob_pattern}')
"""
stats_df = con.sql(stats_query).pl()
print(stats_df)


2. GLOBAL DATASET AGGREGATES
shape: (1, 6)
┌─────────────────┬──────────────┬─────────────────┬──────────────┬──────────────┬─────────────────┐
│ total_documents ┆ avg_char_len ┆ median_char_len ┆ min_char_len ┆ max_char_len ┆ approx_billion_ │
│ ---             ┆ ---          ┆ ---             ┆ ---          ┆ ---          ┆ words           │
│ i64             ┆ f64          ┆ f64             ┆ i64          ┆ i64          ┆ ---             │
│                 ┆              ┆                 ┆              ┆              ┆ f64             │
╞═════════════════╪══════════════╪═════════════════╪══════════════╪══════════════╪═════════════════╡
│ 14868862        ┆ 3082.56      ┆ 1801.0          ┆ 91           ┆ 674682       ┆ 9.167           │
└─────────────────┴──────────────┴─────────────────┴──────────────┴──────────────┴─────────────────┘


In [ ]:
print("\n" + "=" * 60)
print("TOP 10 DOMAINS BY DOCUMENT COUNT")
print("=" * 60)
domain_query = f"""
    SELECT 
        REGEXP_EXTRACT(url, 'https?://([^/]+)', 1) AS domain,
        COUNT(*) AS doc_count
    FROM read_parquet('{glob_pattern}')
    GROUP BY domain
    ORDER BY doc_count DESC
    LIMIT 10
"""
domain_df = con.sql(domain_query).pl()
print(domain_df)


3. TOP 10 DOMAINS BY DOCUMENT COUNT
shape: (10, 2)
┌─────────────────────────────┬───────────┐
│ domain                      ┆ doc_count │
│ ---                         ┆ ---       │
│ str                         ┆ i64       │
╞═════════════════════════════╪═══════════╡
│ en.wikipedia.org            ┆ 15499     │
│ www.theguardian.com         ┆ 12916     │
│ bleacherreport.com          ┆ 9373      │
│ www.prweb.com               ┆ 9071      │
│ www.businessinsider.com     ┆ 6711      │
│ www.washingtonpost.com      ┆ 6555      │
│ www.foxnews.com             ┆ 6443      │
│ www.nytimes.com             ┆ 6406      │
│ www.fanfiction.net          ┆ 6047      │
│ articles.chicagotribune.com ┆ 6022      │
└─────────────────────────────┴───────────┘


In [ ]:
print("\n" + "=" * 60)
print("DOCUMENT DISTRIBUTION BY CRAWL DUMP (Top 5)")
print("=" * 60)
dump_query = f"""
    SELECT 
        dump,
        COUNT(*) AS doc_count
    FROM read_parquet('{glob_pattern}')
    GROUP BY dump
    ORDER BY doc_count DESC
    LIMIT 5
"""
dump_df = con.sql(dump_query).pl()
print(dump_df)


4. DOCUMENT DISTRIBUTION BY CRAWL DUMP (Top 5)
shape: (5, 2)
┌─────────────────┬───────────┐
│ dump            ┆ doc_count │
│ ---             ┆ ---       │
│ str             ┆ i64       │
╞═════════════════╪═══════════╡
│ CC-MAIN-2023-40 ┆ 217633    │
│ CC-MAIN-2022-21 ┆ 215914    │
│ CC-MAIN-2017-17 ┆ 215414    │
│ CC-MAIN-2017-13 ┆ 211543    │
│ CC-MAIN-2023-50 ┆ 211075    │
└─────────────────┴───────────┘


In [ ]:
print("\n" + "=" * 60)
print(f"SAMPLE DOCUMENTS (Showing {num_samples})")
print("=" * 60)
samples_query = f"""
    SELECT id, url, dump, date, LENGTH(text) as char_len, text
    FROM read_parquet('{glob_pattern}')
    LIMIT {num_samples}
"""
samples = con.sql(samples_query).pl().to_dicts()

for idx, sample in enumerate(samples, 1):
    print(f"\n--- [Sample {idx}] ---")
    print(f"ID: {sample['id']}")
    print(f"URL: {sample['url']}")
    print(f"Dump: {sample['dump']} | Date: {sample['date']}")
    print(f"Length: {sample['char_len']} chars")
    print("Text Preview (First 400 chars):")
    print("-" * 40)
    print(
        sample["text"][:400] + ("..." if len(sample["text"]) > 400 else "")
    )
    print("-" * 40)


5. SAMPLE DOCUMENTS (Showing 10)

--- [Sample 1] ---
ID: <urn:uuid:39147604-bfbe-4ed5-b19c-54105f8ae8a7>
URL: http://daytimeroyaltyonline.com/single/?p=8906650&t=8780053
Dump: CC-MAIN-2013-20 | Date: 2013-05-18T05:48:59Z
Length: 414 chars
Text Preview (First 400 chars):
----------------------------------------
|Viewing Single Post From: Spoilers for the Week of February 11th|
|Lil||Feb 1 2013, 09:58 AM|
Don't care about Chloe/Taniel/Jen-Jen. Don't care about Sami, really, but hoping that we get some good "SAMANTHA GENE!!" Marlena Death-Stares out of it. And "newfound" feelings. Please. If only.
STEFANO!! STEFANO, STEFANO, STEFANO!!!! :cheer:
|Spoilers for the Week of February 11th · DAYS: News, Spoilers...
----------------------------------------

--- [Sample 2] ---
ID: <urn:uuid:ba819eb7-e6e6-415a-87f4-0347b6a4f017>
URL: http://endogenousretrovirus.blogspot.com/2007/11/if-you-have-set-yourself-on-fire-do-not.html?showComment=1196270520000
Dump: CC-MAIN-2013-20 | Date: 2013-05-18T06:4